# 02. Generate IHO Region Masks

Generate the regional masks on the processed OISST grid.

EAMS: Yellow Sea (YS), East China Sea (ECS), and East/Japan Sea (EJS).
Philippine Sea (PHS) is retained separately.

Download **IHO Sea Areas — Version 3 (2018)** and place it under `data/external/World_Seas_IHO_v3/`.

In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

def find_repo_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "configs" / "manuscript.yml").exists() and (candidate / "src" / "eams_seof").exists():
            return candidate
    raise FileNotFoundError("Repository root not found.")


REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from eams_seof.config import load_config, resolve_path

CONFIG_PATH = REPO_ROOT / "configs" / "manuscript.yml"
cfg = load_config(CONFIG_PATH, repo_root=REPO_ROOT)

print("Repository:", REPO_ROOT)
print("Config:", CONFIG_PATH)


In [ ]:
import xarray as xr
import matplotlib.pyplot as plt

from eams_seof.regions import build_region_masks

sst_path = resolve_path(REPO_ROOT, cfg["paths"]["oisst_smoothed"])
shp_path = resolve_path(REPO_ROOT, cfg["paths"]["iho_shapefile"])
mask_path = resolve_path(REPO_ROOT, cfg["paths"]["region_masks"])

sst = xr.open_dataset(sst_path)["sst"]

lat_bounds = cfg["domain"]["latitude"]
lon_bounds = cfg["domain"]["longitude"]

domain = sst.sel(
    lat=slice(*lat_bounds),
    lon=slice(*lon_bounds),
)

print(domain.lon.values[[0, -1]])
print(domain.lat.values[[0, -1]])


In [ ]:
region_masks = build_region_masks(
    lon=domain.lon,
    lat=domain.lat,
    shapefile=shp_path,
)

mask_path.parent.mkdir(parents=True, exist_ok=True)
region_masks.to_netcdf(mask_path)

print("Saved:", mask_path)
display(region_masks)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

region_masks["region_mask"].plot(
    ax=ax,
    robust=True,
)

ax.set_title("IHO regional masks on the OISST grid")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

plt.show()